In [ ]:
import requests
from tabulate import tabulate  # pip install tabulate (или уберите этот импорт и print ниже)

# Список моделей с сервера
MODELS = [
    "smollm2:135m", "qwen3:8b", "mistral-small3.1:24b", "mixtral:8x7b", 
    "gpt-oss:20b", "gpt-oss:120b", "llama4:16x17b", "granite3.2-vision:2b", 
    "gemma3:27b", "mistral-small3.1-24b-128k:latest", "mistral-small:24b", 
    "deepseek-v2:16b", "qwen3:32b", "llama3:70b", "deepseek-r1:70b", 
    "qwen2.5vl:72b", "mistral-small3.1:latest", "llama4:latest"
]

BASE_URL = "https://aicltr.itmo.ru/ollama/api/show"

def get_model_info(model_name):
    try:
        response = requests.post(BASE_URL, json={"model": model_name}, timeout=10)
        if response.status_code != 200:
            return None
        data = response.json()
        
        info = data.get("model_info", {})
        details = data.get("details", {})
        
        # Ищем длину контекста в разных ключах (зависит от архитектуры)
        ctx = info.get("qwen3.context_length") or \
              info.get("llama.context_length") or \
              info.get("gptoss.context_length") or \
              info.get("general.context_length") or \
              "Unknown"
              
        return {
            "Model": model_name,
            "Size": details.get("parameter_size", "N/A"),
            "Quant": details.get("quantization_level", "N/A"),
            "Context Limit": ctx,
            "Family": details.get("family", "N/A")
        }
    except Exception as e:
        return {"Model": model_name, "Size": "Error", "Context Limit": str(e)}

print(f"{'Model':<40} | {'Size':<10} | {'Quant':<10} | {'Context':<10} | {'Family'}")
print("-" * 90)

for model in MODELS:
    info = get_model_info(model)
    if info:
        print(f"{info['Model']:<40} | {info['Size']:<10} | {info['Quant']:<10} | {str(info['Context Limit']):<10} | {info.get('Family', '')}")

Model                                    | Size       | Quant      | Context    | Family
------------------------------------------------------------------------------------------
smollm2:135m                             | 134.52M    | F16        | 8192       | llama
qwen3:8b                                 | 8.2B       | Q4_K_M     | 40960      | qwen3
mistral-small3.1:24b                     | 24.0B      | Q4_K_M     | Unknown    | mistral3
mixtral:8x7b                             | 46.7B      | Q4_0       | 32768      | llama
gpt-oss:20b                              | 20.9B      | MXFP4      | 131072     | gptoss
gpt-oss:120b                             | 116.8B     | MXFP4      | 131072     | gptoss
llama4:16x17b                            | 108.6B     | Q4_K_M     | Unknown    | llama4
granite3.2-vision:2b                     | 2.5B       | Q4_K_M     | Unknown    | granite
gemma3:27b                               | 27.4B      | Q4_K_M     | Unknown    | gemma3
mistral-small3.1-24